<a href="https://colab.research.google.com/github/Ali-Shahrez/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

## 1. Ranked actions + reason codes

This queue is built on the honest Random Forest from ML-08/ML-09 (client-holdout split,
eligibility gate `impressions_90d >= 500`), retrained here from the same canonical split so
the notebook is self-contained.

**Population scored:** the full eligible population (train + test, ~16,726 rows) — not just
the 618-row test set. Every row carries a `validation_status` flag:
- `out_of_sample` — the ~618 test-eligible rows. RF score is genuinely honest here (this is
  the number ML-09 audited).
- `in_sample_fit` — the ~16,108 train-eligible rows. The model was fit on these rows, so the
  score reflects fit quality, not validated generalization — ML-08 showed train P@50 = 1.000,
  which is an overfitting signature, not a confidence signal. These scores are directional at
  best and are labeled as such everywhere they appear in this notebook and in the export.

Reason codes are built only from features that were in the honest feature set — nothing
derived from `trend_direction`/`trend_pct`. Multiple reason codes can apply to one row; the
combination is what determines the action (see the markdown write-up after the code, once we
see real numbers).

In [37]:
%cd /content
!rm -rf flyrank-ml-internship
!git clone -q https://github.com/Ali-Shahrez/flyrank-ml-internship.git
%cd flyrank-ml-internship

import numpy as np
import pandas as pd

RANDOM_STATE = 42
MIN_IMPRESSIONS = 500

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(f"Loaded {len(df):,} rows")

# Canonical client-holdout split — identical method to w05/w06
client_series = df['client_id'].fillna('unknown').astype(str)
unique_clients = client_series.drop_duplicates().to_numpy()
rng = np.random.default_rng(RANDOM_STATE)
shuffled_clients = rng.permutation(unique_clients)
test_client_count = max(1, int(round(len(shuffled_clients) * 0.2)))
test_clients = set(shuffled_clients[:test_client_count])
test_mask = client_series.isin(test_clients).to_numpy()

train_df = df[~test_mask].copy()
test_df  = df[test_mask].copy()

train_eligible = train_df[train_df['impressions_90d'] >= MIN_IMPRESSIONS].copy()
test_eligible  = test_df[test_df['impressions_90d'] >= MIN_IMPRESSIONS].copy()
for f in (train_eligible, test_eligible):
    f['is_declining_label'] = (f['trend_direction'] == 'down').astype(int)

train_eligible['validation_status'] = 'in_sample_fit'
test_eligible['validation_status']  = 'out_of_sample'

print(f"Train eligible: {len(train_eligible):,} | Test eligible: {len(test_eligible):,}")
assert len(train_eligible) == 16108 and len(test_eligible) == 618, \
    "Split doesn't match the documented canonical numbers — stop and check before continuing"
print("Split verified against documented canonical numbers.")

/content
/content/flyrank-ml-internship
Loaded 30,000 rows
Train eligible: 16,108 | Test eligible: 618
Split verified against documented canonical numbers.


In [38]:
import sys, os
sys.path.append(os.path.abspath('scripts'))
from ml_utils import precision_at_k, MODEL_NUMERIC_FEATURES, MODEL_CATEGORICAL_FEATURES
from sklearn.ensemble import RandomForestClassifier

MISSINGNESS_TRACKED_COLS = ['word_count', 'char_count', 'search_volume', 'competition', 'cpc']

def add_derived_and_flags(frame):
    frame = frame.copy()
    for col in MISSINGNESS_TRACKED_COLS:
        frame[f'has_{col}'] = frame[col].notna().astype(int)
    frame['log_impressions_90d'] = np.log1p(frame['impressions_90d'])
    frame['log_clicks_90d']      = np.log1p(frame['clicks_90d'])
    frame['log_sessions_90d']    = np.log1p(frame['sessions_90d'])
    frame['log_ai_sessions_90d'] = np.log1p(frame['ai_sessions_90d'])
    return frame

def build_features(frame, numeric_cols, categorical_cols, flag_cols):
    numeric = frame[numeric_cols].apply(pd.to_numeric, errors='coerce').replace([np.inf, -np.inf], np.nan).fillna(0)
    flags = frame[flag_cols]
    categorical = frame[categorical_cols].fillna('unknown').astype(str)
    dummies = pd.get_dummies(categorical, prefix=categorical_cols, dtype=float)
    return pd.concat([numeric.reset_index(drop=True), flags.reset_index(drop=True), dummies.reset_index(drop=True)], axis=1)

train_eligible = add_derived_and_flags(train_eligible)
test_eligible  = add_derived_and_flags(test_eligible)

numeric_cols = [c for c in MODEL_NUMERIC_FEATURES if c in train_eligible.columns]
categorical_cols = [c for c in MODEL_CATEGORICAL_FEATURES if c in train_eligible.columns]
flag_cols = [f'has_{c}' for c in MISSINGNESS_TRACKED_COLS]

X_train = build_features(train_eligible, numeric_cols, categorical_cols, flag_cols)
X_test  = build_features(test_eligible,  numeric_cols, categorical_cols, flag_cols)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

y_train = train_eligible['is_declining_label']
y_test  = test_eligible['is_declining_label']

assert not any(c in X_train.columns for c in ['trend_direction', 'trend_pct', 'content_id', 'client_id']), \
    "Leakage check failed"

rf = RandomForestClassifier(class_weight="balanced_subsample", max_depth=10,
                             min_samples_leaf=25, n_estimators=200, n_jobs=-1, random_state=RANDOM_STATE)
rf.fit(X_train, y_train)

train_eligible['rf_score'] = rf.predict_proba(X_train)[:, 1]
test_eligible['rf_score']  = rf.predict_proba(X_test)[:, 1]

print(f"Out-of-sample P@20: {precision_at_k(y_test, test_eligible['rf_score'], 20):.3f}")
print(f"Out-of-sample P@50: {precision_at_k(y_test, test_eligible['rf_score'], 50):.3f}")
print(f"In-sample (fit) P@20: {precision_at_k(y_train, train_eligible['rf_score'], 20):.3f}")
print(f"In-sample (fit) P@50: {precision_at_k(y_train, train_eligible['rf_score'], 50):.3f}")

Out-of-sample P@20: 0.850
Out-of-sample P@50: 0.860
In-sample (fit) P@20: 1.000
In-sample (fit) P@50: 1.000


In [39]:
status_order = {'out_of_sample': 0, 'in_sample_fit': 1}
queue['status_rank'] = queue['validation_status'].map(status_order)

queue_ranked = queue.sort_values(
    by=['status_rank', 'rf_score'],
    ascending=[True, False]
).drop(columns='status_rank').reset_index(drop=True)

print("Top 20 validation_status counts (should be all out_of_sample now):")
print(queue_ranked.head(20)['validation_status'].value_counts())
print()

cols = ['content_id', 'client_id', 'rf_score', 'validation_status', 'action', 'reason_codes',
        'avg_position', 'ctr', 'impressions_90d', 'engagement_rate', 'scroll_rate']
queue_ranked[cols].head(20)

Top 20 validation_status counts (should be all out_of_sample now):
validation_status
out_of_sample    20
Name: count, dtype: int64



,content_id,client_id,rf_score,validation_status,action,reason_codes,avg_position,ctr,impressions_90d,engagement_rate,scroll_rate
0,content_6e792cf3ce56,client_f74efabef1,0.780279,out_of_sample,refresh_content,"MODEL_DECLINE_RISK,RANKING_SLIPPED",29.1,0.06,4908,0.00,16.67
1,content_0cf67ec37ab8,client_f74efabef1,0.776617,out_of_sample,investigate_external,"MODEL_DECLINE_RISK,RANKING_INTACT_DEMAND_DROP",3.0,0.00,767,0.00,50.00
2,content_331182ca4cae,client_f74efabef1,0.751619,out_of_sample,refresh_content,"MODEL_DECLINE_RISK,RANKING_SLIPPED",35.9,0.00,3026,0.00,0.00
3,content_52b1c884e871,client_f74efabef1,0.750648,out_of_sample,review_metadata,"MODEL_DECLINE_RISK,WEAK_CTR_FOR_POSITION",15.1,0.15,682,0.00,40.00
4,content_818c81a114e4,client_f74efabef1,0.745253,out_of_sample,review_metadata,"MODEL_DECLINE_RISK,WEAK_CTR_FOR_POSITION",15.1,0.00,1243,0.00,0.00
5,content_d603c0b7de2e,client_f74efabef1,0.744184,out_of_sample,refresh_content,"MODEL_DECLINE_RISK,RANKING_SLIPPED",23.3,0.00,554,0.00,0.00
6,content_89f63af999b1,client_f74efabef1,0.743108,out_of_sample,refresh_content,"MODEL_DECLINE_RISK,RANKING_SLIPPED",28.6,0.00,901,0.00,0.00
7,content_1df2423841db,client_f74efabef1,0.738269,out_of_sample,investigate_external,"MODEL_DECLINE_RISK,RANKING_INTACT_DEMAND_DROP",9.4,0.00,1343,0.00,0.00
8,content_fded62af1b93,client_f74efabef1,0.736958,out_of_sample,investigate_external,"MODEL_DECLINE_RISK,RANKING_INTACT_DEMAND_DROP",7.1,0.00,547,0.00,0.00
9,content_00603b0349b4,client_f74efabef1,0.732427,out_of_sample,refresh_content,"MODEL_DECLINE_RISK,RANKING_SLIPPED",25.6,0.09,1076,11.11,22.22


### Reading the queue

**Sort order matters and is itself a claim.** The queue is ordered `validation_status` first
(`out_of_sample` before `in_sample_fit`), `rf_score` second. Sorting by score alone would have
put in-sample rows at the top — rows the model was fit on, where train Precision@50 = 1.000
(a documented overfitting signature from ML-08, not evidence of quality). Every row a reviewer
sees first is genuinely out-of-sample.

**Action counts, out-of-sample only (n=618, the trustworthy tier):**

| Action | Count |
|---|---:|
| no_action | 414 |
| review_metadata | 83 |
| investigate_external | 76 |
| refresh_content | 45 |
| review_onpage_engagement | 0 |
| monitor_closely | 0 |

Two actions never fire in this tier. That is reported as-is, not treated as proof those
categories are rare in general — with n=618 and ~97% of it one client (per ML-08/ML-09), an
absence here is weak evidence of anything beyond "not present in this particular slice."

**Top 20 (all out-of-sample) are entirely `client_f74efabef1`.** This restates, on a new
artifact, the same limitation ML-08 and ML-09 already established for this test split: it is
effectively a single-client holdout. The queue's top recommendations describe one client's
content in depth, not a cross-client sample. Reading this queue as "the top 20 pages across
the portfolio" would overstate what this test split can support.

**`RANKING_INTACT_DEMAND_DROP` needs a precise reading.** The code fires this reason code on
`avg_position` alone (0 < position <= 10); it does not additionally check that CTR or
engagement are healthy. Several top-20 rows carry this code with `ctr = 0.00` — "ranking
intact" means the position component specifically, not a clean bill of health on every metric.
The reason code name is being tightened in a future pass to avoid this ambiguity; noted here so
the current output isn't over-read in the meantime.

**In-sample rows (`validation_status = in_sample_fit`) are directional only.** They make up
16,108 of the queue's 16,726 rows and are not held out to a level that Precision@K can
honestly certify — they are included because a queue of 618 rows understates what a real
review workflow needs, not because their scores carry the same evidence as the out-of-sample
tier's.

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

In [40]:
below_gate = (df['impressions_90d'] < MIN_IMPRESSIONS).sum()
total = len(df)
print(f"Pages below the eligibility gate (no recommendation at all): {below_gate:,} of {total:,} ({below_gate/total:.1%})")

Pages below the eligibility gate (no recommendation at all): 13,274 of 30,000 (44.2%)


## 2. Intended use and limits

**Who uses this:** a content strategist or SEO editor with limited weekly review capacity,
deciding which pages to look at first — not an automated publishing or editing system. Every
action in this queue is a suggestion for a human to check, per ML-02's original framing: the
cost of a wrong call runs in both directions (wasted editor time vs. a real client losing
traffic), which is why this stays a ranking/prioritization tool, not a yes/no verdict.

**What "intended use" does NOT include:**
- Auto-applying any action (refresh, metadata edit, etc.) without a person opening the page
- Treating `MODEL_DECLINE_RISK` as proof a page is declining — it is the proxy label
  (`trend_direction == "down"`) the model was trained to predict, and that label itself is a
  same-window threshold rule, not an observed future outcome (flagged as a limitation since
  ML-03 and never resolved on this dataset)
- Comparing scores *across* the `validation_status` tiers as if they mean the same thing —
  they don't (Section 1)
- Using this queue as evidence that refreshing a page *causes* recovery — nothing in this
  pipeline supports a causal claim (see ML-09's audit of the FlyRank paper's own refresh
  finding, which hit the same wall)

**Where it stops being valid:**

1. **Pages below the eligibility gate get no recommendation at all.** `impressions_90d >= 500`
   excludes 44.2% of the full 30,000-row dataset (13,274 pages) — low-traffic pages are
   invisible to this playbook entirely, not scored as low-priority. Nearly half the portfolio
   sits outside this tool's coverage. A separate, lower-volume-appropriate process would be
   needed for them; this one doesn't claim to cover them.
2. **The label is a same-window proxy, not a future outcome.** `is_declining_label` compares
   the current 90-day window's trend to itself — it was never validated against what actually
   happened to a page afterward. Every action in this queue is downstream of that same
   limitation.
3. **Out-of-sample validation covers 618 rows, ~97% one client.** The only genuinely-tested
   precision numbers (P@20=0.85, P@50=0.86) describe `client_f74efabef1`'s content specifically.
   Extending that confidence to other clients or to the full portfolio is not supported by this
   test design.
4. **The model can't distinguish a currently-declining page from a currently-recovering one
   that happens to share the same weak-position/low-CTR/stale profile.** ML-09's error analysis
   found false positives and false negatives sitting on nearly identical score bands (~0.70–0.75)
   — the one feature that would resolve the ambiguity, recent trend direction, is exactly the
   column the leakage audit correctly excludes. This is a structural ceiling of the honest
   feature set, not a bug a retrain would fix.
5. **Reason codes rely partly on a synthetic-data artifact.** `STALE_UPDATE` is built on
   `freshness_tier`, and ML-07 found the underlying `days_since_last_update` column carries
   heavy tie-mass at specific values (104 in the full population, 20 in this test split) that
   are almost certainly synthetic-batch artifacts, not real staleness variation. Treat
   `STALE_UPDATE` as the weakest reason code in the set.
6. **Nothing here is validated on the full warehouse.** This entire playbook is scoped to the
   30K-row anonymized CSV slice. It is a decision-support prototype proven on one dataset, not
   a production system.

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

In [41]:
def assign_action(row):
    if not row['MODEL_DECLINE_RISK']:
        return 'no_action', []
    codes = []
    if row['RANKING_SLIPPED']:
        codes.append('RANKING_SLIPPED')
        action = 'refresh_content'
    elif row['RANKING_INTACT_DEMAND_DROP']:
        codes.append('RANKING_INTACT_DEMAND_DROP')
        action = 'investigate_external'
    elif row['WEAK_CTR_FOR_POSITION']:
        codes.append('WEAK_CTR_FOR_POSITION')
        action = 'review_metadata'
    elif row['LOW_ENGAGEMENT']:
        codes.append('LOW_ENGAGEMENT')
        action = 'review_onpage_engagement'
    else:
        action = 'monitor_closely'
    if row['STALE_UPDATE']:
        codes.append('STALE_UPDATE')  # never drives the action alone, only rides along
    return action, codes

results = queue.apply(assign_action, axis=1, result_type='expand')
queue['action'] = results[0]
queue['reason_codes'] = results[1].apply(lambda cs: ','.join(['MODEL_DECLINE_RISK'] + cs))
queue.loc[queue['action'] == 'no_action', 'reason_codes'] = ''

print(queue[queue['MODEL_DECLINE_RISK']]['reason_codes'].str.contains('STALE_UPDATE').sum(),
      "flagged rows now carry STALE_UPDATE as a secondary code")

1787 flagged rows now carry STALE_UPDATE as a secondary code


In [42]:
status_order = {'out_of_sample': 0, 'in_sample_fit': 1}
queue['status_rank'] = queue['validation_status'].map(status_order)
queue_ranked = queue.sort_values(
    by=['status_rank', 'rf_score'], ascending=[True, False]
).drop(columns='status_rank').reset_index(drop=True)

In [43]:
flagged_queue = queue_ranked[queue_ranked['action'] != 'no_action']
print("Client distribution across all flagged (actionable) rows:")
print(flagged_queue['client_id'].value_counts().head(10))
print(f"\nTotal flagged rows: {len(flagged_queue):,}")
print(f"Distinct clients represented: {flagged_queue['client_id'].nunique()}")

Client distribution across all flagged (actionable) rows:
client_id
client_3fdba35f04    1106
client_6208ef0f77     926
client_7f2253d7e2     865
client_19581e27de     333
client_f369cb89fc     218
client_f74efabef1     202
client_349c41201b     149
client_a88a7902cb     130
client_4e07408562      99
client_bbb965ab0c      79
Name: count, dtype: int64

Total flagged rows: 4,290
Distinct clients represented: 23


## 3. Human review + the no-go list

**No-go list — never automate these:**

- **Auto-applying any action without a human opening the page.** Every recommendation here is
  a suggestion, not a decision. This is the hard version of the framing in Section 2.
- **Treating a single flagged page as certain evidence of decline.** ML-09's error analysis
  found the model scores currently-recovering and currently-declining pages in the same band
  (~0.70–0.75) when they share the same weak-position/low-CTR/stale profile — a flag is a
  reason to look, not a verdict.
- **Using `in_sample_fit` scores to justify urgency or sequencing.** That tier has no held-out
  evidence behind its ordering (Section 1); it should inform *what* to check, not *how fast*.
- **Acting on `STALE_UPDATE` alone.** It never drives an action on its own in the code — it
  only rides along as a secondary tag (1,787 rows carry it) — and per Section 2 it's partly
  built on a documented synthetic-data artifact in `days_since_last_update`.
- **Generalizing the 0.85/0.86 precision numbers to any client other than
  `client_f74efabef1`.** That's the client the out-of-sample test set actually measured.
- **Treating `no_action` as "this page is fine."** 44.2% of the full dataset never entered
  scoring (below the eligibility gate) and isn't represented in `no_action` at all. Within the
  scored population, `no_action` means "not flagged," not "verified healthy."

**Human review checklist, before acting on any flagged row:**

1. Open the page. Does the reason code match what's actually there — e.g. does
   `WEAK_CTR_FOR_POSITION` reflect a real snippet/title problem, or is `ctr = 0.00` just a
   low-volume artifact?
2. Check `validation_status`. Out-of-sample scores carry real evidence; `in_sample_fit` scores
   are directional only.
3. For `RANKING_INTACT_DEMAND_DROP` rows specifically: this code fires on `avg_position` alone
   (Section 1). Confirm CTR and engagement are also holding up before assuming an external
   cause (seasonality, consolidation) is the right read — a weak-CTR page with intact position
   is a different problem than a genuinely healthy one.
4. Check which client the page belongs to. Flagged rows span 23 distinct clients, but 68%
   (2,897 of 4,290) sit in just three (`client_3fdba35f04`, `client_6208ef0f77`,
   `client_7f2253d7e2`) — for a client outside that top group, or outside the out-of-sample
   test population entirely, treat the score as less proven regardless of its
   `validation_status` label, since the model's real-world track record is concentrated in a
   handful of clients.
5. Rule out the lookalikes from the lane guide before committing effort: consolidation
   (a sibling page absorbing demand), seasonality, and SERP/AI click loss can all produce the
   same observable pattern as a genuine decline, and none of them are fixed by a content
   refresh.

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

This is a decision-support prototype on a static CSV snapshot, not a live system — there's no
production pipeline pushing new data through it. "Monitoring" here means: what would tell a
future version of this project the current numbers have gone stale or were never trustworthy
enough to begin with.

**Retrain / rebuild triggers:**

1. **Warehouse rebuild with a genuine future-window label.** This entire playbook is built on
   `is_declining_label`, a same-window proxy (Section 2, point 2). The moment a future-window
   label (prior 90 days → next 30 days, per ML-04/ML-09) is validated on the warehouse, that
   model supersedes this one — not as an incremental update, as a replacement. This is the
   single most important trigger on this list.
2. **Schema mismatch blocks any warehouse scoring.** As established this session: the current
   model's feature contract (`MODEL_NUMERIC_FEATURES`/`MODEL_CATEGORICAL_FEATURES`) doesn't
   exist in the warehouse's column names or grain. This model cannot be pointed at warehouse
   rows without a full feature-pipeline rebuild — that rebuild, when it happens, is a retrain
   trigger, not a scoring option for the current model.
3. **A lower-volume-appropriate method for the 44.2% excluded population.** If one gets built,
   it's a genuine expansion of coverage, not a patch to this model — the eligibility gate exists
   because the current feature set is unreliable below it (Section 2, point 1), and that
   reasoning doesn't go away by adding a second model.

**Drift checks, if this were re-run on a new export:**

- **Test-set base rate moves materially from 0.519.** ML-08 already showed base rate isn't
  stable across splits (0.542 dataset-wide vs. 0.391 vs. 0.519 across different holdouts) — a
  large shift signals the population changed enough that current precision numbers may not
  transfer.
- **The `days_since_last_update == 104` (or `==20`) tie-mass shrinks or disappears.** Per ML-07,
  this is flagged as a synthetic-data artifact. If a future export doesn't reproduce it, that's
  a sign the data generation changed — worth re-running the leakage/artifact checks from
  scratch rather than assuming old findings still hold.
- **Client concentration in the out-of-sample tier changes.** The current 0.85/0.86 numbers
  describe `client_f74efabef1` specifically (598/618 test rows). A re-split that spreads test
  rows across more clients would need fresh precision numbers before reusing any claim from
  this notebook.
- **Flagged-row client distribution shifts.** Baseline for comparison: 23 distinct clients in
  the flagged population, top 3 clients = 68% of flagged rows. A future run concentrating much
  more narrowly (or spreading much more evenly) is worth a second look before trusting the
  queue's action mix.

**Review cadence (practical, non-production):** given this is a static teaching dataset with no
live refresh cycle, there's no meaningful automated cadence to define. If this pipeline were
ever pointed at a live, updating export, a monthly re-check of the four drift signals above
against this notebook's baseline numbers would be a reasonable starting cadence — tightened or
loosened once real drift behavior is observed, not fixed in advance.

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [44]:
import os
import json

os.makedirs('work/outputs', exist_ok=True)
os.makedirs('work/figures', exist_ok=True)

# Coarse confidence display for in-sample rows — no false precision on unvalidated scores
queue_ranked['rf_confidence_display'] = queue_ranked.apply(
    lambda r: round(r['rf_score'], 3) if r['validation_status'] == 'out_of_sample'
              else ('flagged' if r['MODEL_DECLINE_RISK'] else 'not_flagged'),
    axis=1
)

export_cols = [
    'content_id', 'client_id', 'validation_status', 'rf_confidence_display',
    'action', 'reason_codes', 'avg_position', 'ctr', 'impressions_90d',
    'engagement_rate', 'scroll_rate', 'days_since_last_update'
]
queue_export = queue_ranked[export_cols]
queue_export.to_csv('work/outputs/action_playbook_queue.csv', index=False)
print(f"Exported {len(queue_export):,} rows to work/outputs/action_playbook_queue.csv")

Exported 16,726 rows to work/outputs/action_playbook_queue.csv


In [45]:
import sys
sys.path.append(os.path.abspath('scripts'))
from ml_utils import simple_svg_bar_chart
from pathlib import Path

action_mix_oos = queue_ranked[queue_ranked['validation_status'] == 'out_of_sample']['action'].value_counts()

simple_svg_bar_chart(
    title="Action mix — out-of-sample (validated) rows only, n=618",
    labels=list(action_mix_oos.index),
    values=list(action_mix_oos.values),
    path=Path('work/figures/action_mix_out_of_sample.svg'),
)
print("Saved work/figures/action_mix_out_of_sample.svg")

Saved work/figures/action_mix_out_of_sample.svg


In [46]:
metrics = {
    "model": "random_forest",
    "canonical_split": {
        "train_eligible": len(train_eligible),
        "test_eligible": len(test_eligible),
        "test_client_dominant": "client_f74efabef1",
        "test_client_dominant_share": round(598/618, 3),
    },
    "precision_out_of_sample": {
        "p20": round(precision_at_k(y_test, test_eligible['rf_score'], 20), 3),
        "p50": round(precision_at_k(y_test, test_eligible['rf_score'], 50), 3),
    },
    "precision_in_sample_fit": {
        "p20": round(precision_at_k(y_train, train_eligible['rf_score'], 20), 3),
        "p50": round(precision_at_k(y_train, train_eligible['rf_score'], 50), 3),
        "note": "overfitting signature, not a validated confidence number",
    },
    "eligibility_gate": {
        "threshold": MIN_IMPRESSIONS,
        "excluded_rows": int(below_gate),
        "excluded_pct": round(below_gate / total, 3),
    },
    "flagged_queue": {
        "total_flagged": int(len(flagged_queue)),
        "distinct_clients": int(flagged_queue['client_id'].nunique()),
        "top3_client_share": round(flagged_queue['client_id'].value_counts().head(3).sum() / len(flagged_queue), 3),
        "action_counts_out_of_sample": action_mix_oos.to_dict(),
    },
    "known_limitations": [
        "same-window proxy label, not a future outcome",
        "out-of-sample precision describes one dominant client (client_f74efabef1)",
        "STALE_UPDATE reason code partly reflects a synthetic-data tie-mass artifact",
        "model cannot distinguish declining vs. recovering pages sharing the same feature profile",
        "not validated on the full warehouse; current feature contract is CSV-schema-specific",
    ],
}

with open('work/outputs/action_playbook_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print("Wrote work/outputs/action_playbook_metrics.json")
print(json.dumps(metrics, indent=2))

Wrote work/outputs/action_playbook_metrics.json
{
  "model": "random_forest",
  "canonical_split": {
    "train_eligible": 16108,
    "test_eligible": 618,
    "test_client_dominant": "client_f74efabef1",
    "test_client_dominant_share": 0.968
  },
  "precision_out_of_sample": {
    "p20": 0.85,
    "p50": 0.86
  },
  "precision_in_sample_fit": {
    "p20": 1.0,
    "p50": 1.0,
    "note": "overfitting signature, not a validated confidence number"
  },
  "eligibility_gate": {
    "threshold": 500,
    "excluded_rows": 13274,
    "excluded_pct": 0.442
  },
  "flagged_queue": {
    "total_flagged": 4290,
    "distinct_clients": 23,
    "top3_client_share": 0.675,
    "action_counts_out_of_sample": {
      "no_action": 414,
      "review_metadata": 83,
      "investigate_external": 76,
      "refresh_content": 45
    }
  },
  "known_limitations": [
    "same-window proxy label, not a future outcome",
    "out-of-sample precision describes one dominant client (client_f74efabef1)",
    "ST

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.